In [ ]:
from pyspark.sql import SparkSession
import time

from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier,
    LinearSVC
)

from pyspark.ml.tuning import (
    ParamGridBuilder,
    TrainValidationSplit
)

from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

# Creating Spark Session

In [87]:
spark = SparkSession.builder \
    .appName("AmazonReviews_Task3") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.0.3


# Loading Training and Testing Data

In [ ]:
train_df = spark.read.parquet("train_df")
test_df = spark.read.parquet("test_df")

# Verifying Dataset

In [ ]:
print("Training Rows :", train_df.count())
print("Testing Rows :", test_df.count())

print("Training Columns :", train_df.columns)
print("Testing Columns :", test_df.columns)

Training Rows : 8799451
Testing Rows : 2200549
Training Columns : ['helpful_vote', 'verified_purchase', 'label', 'features', 'scaledFeatures']
Testing Columns : ['helpful_vote', 'verified_purchase', 'label', 'features', 'scaledFeatures']


# Creating Evaluators

In [ ]:
binaryEvaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

accuracyEvaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

precisionEvaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recallEvaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

f1Evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

# Model 1 – Logistic Regression

In [ ]:
lr = LogisticRegression(
    featuresCol="scaledFeatures",
    labelCol="label"
)

## Hyperparameter Tuning

In [ ]:
paramGrid_lr = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.01, 0.1])
    .addGrid(lr.maxIter, [10, 20])
    .build()
)

In [ ]:
tvs_lr = TrainValidationSplit(
    estimator=lr,
    estimatorParamMaps=paramGrid_lr,
    evaluator=binaryEvaluator,
    trainRatio=0.8
)

In [ ]:
# Logistic Regression Training Model
start = time.time()

lr_model = tvs_lr.fit(train_df)

end = time.time()

lr_training_time = end - start

print(f"Training Time: {lr_training_time:.2f} seconds")

Training Time: 243.82 seconds


## Best Parameters

In [ ]:
best_lr = lr_model.bestModel

print("Best Regularization:", best_lr.getRegParam())
print("Best Max Iterations:", best_lr.getMaxIter())

Best Regularization: 0.01
Best Max Iterations: 10


## Predictions

In [ ]:
lr_predictions = lr_model.transform(test_df)

## Accuracy

In [ ]:
accuracy_lr = accuracyEvaluator.evaluate(lr_predictions)

print("Accuracy =", accuracy_lr)

Accuracy = 0.8485991450315353


## Precision

In [ ]:
precision_lr = precisionEvaluator.evaluate(lr_predictions)

print("Precision =", precision_lr)

Precision = 0.7788250874637537


## Recall

In [ ]:
recall_lr = recallEvaluator.evaluate(lr_predictions)

print("Recall =", recall_lr)

Recall = 0.8485991450315353


## F1 Score

In [ ]:
f1_lr = f1Evaluator.evaluate(lr_predictions)

print("F1 Score =", f1_lr)

F1 Score = 0.7799379708262327


## AUC

In [ ]:
auc_lr = binaryEvaluator.evaluate(lr_predictions)

print("AUC =", auc_lr)

AUC = 0.5988158072471639


## Sample Predictions

In [ ]:
lr_predictions.select(
    "label",
    "prediction",
    "probability"
).show(10, truncate=False)

+-----+----------+----------------------------------------+
|label|prediction|probability                             |
+-----+----------+----------------------------------------+
|1    |1.0       |[0.16596159748413902,0.8340384025158609]|
|0    |1.0       |[0.176028018815222,0.823971981184778]   |
|0    |1.0       |[0.176028018815222,0.823971981184778]   |
|0    |1.0       |[0.176028018815222,0.823971981184778]   |
|0    |1.0       |[0.176028018815222,0.823971981184778]   |
|0    |1.0       |[0.176028018815222,0.823971981184778]   |
|0    |1.0       |[0.176028018815222,0.823971981184778]   |
|0    |1.0       |[0.176028018815222,0.823971981184778]   |
|0    |1.0       |[0.176028018815222,0.823971981184778]   |
|0    |1.0       |[0.176028018815222,0.823971981184778]   |
+-----+----------+----------------------------------------+
only showing top 10 rows


# Model 2 – Decision Tree Classifier

## Create Decision Tree Model

In [ ]:
dt = DecisionTreeClassifier(
    featuresCol="scaledFeatures",
    labelCol="label",
    seed=42
)

## Hyperparameter Tuning

In [ ]:
paramGrid_dt = (
    ParamGridBuilder()
    .addGrid(dt.maxDepth, [5, 10])
    .addGrid(dt.maxBins, [32, 64])
    .build()
)

In [ ]:
tvs_dt = TrainValidationSplit(
    estimator=dt,
    estimatorParamMaps=paramGrid_dt,
    evaluator=binaryEvaluator,
    trainRatio=0.8
)

## Train Model

In [ ]:
# Decision Tree Classifier

start = time.time()

dt_model = tvs_dt.fit(train_df)

end = time.time()

dt_training_time = end - start

print(f"Training Time: {dt_training_time:.2f} seconds")

Training Time: 591.51 seconds


## Best Hyperparameters

In [ ]:
best_dt = dt_model.bestModel

print("Best Max Depth :", best_dt.getMaxDepth())
print("Best Max Bins :", best_dt.getMaxBins())

Best Max Depth : 5
Best Max Bins : 32


## Predict Test Data

In [ ]:
dt_predictions = dt_model.transform(test_df)

## Accuracy

In [ ]:
accuracy_dt = accuracyEvaluator.evaluate(dt_predictions)

print("Accuracy =", accuracy_dt)

Accuracy = 0.8488031850233737


## Precision

In [ ]:
precision_dt = precisionEvaluator.evaluate(dt_predictions)

print("Precision =", precision_dt)

Precision = 0.7204668469058235


## Recall

In [ ]:
recall_dt = recallEvaluator.evaluate(dt_predictions)

print("Recall =", recall_dt)

Recall = 0.8488031850233737


## F1 Score

In [ ]:
f1_dt = f1Evaluator.evaluate(dt_predictions)

print("F1 Score =", f1_dt)

F1 Score = 0.7793872844249942


## AUC

In [ ]:
auc_dt = binaryEvaluator.evaluate(dt_predictions)

print("AUC =", auc_dt)

AUC = 0.5


## Sample Predictions

In [ ]:
dt_predictions.select(
    "label",
    "prediction",
    "probability"
).show(10, truncate=False)

+-----+----------+---------------------------------------+
|label|prediction|probability                            |
+-----+----------+---------------------------------------+
|1    |1.0       |[0.1509902151850155,0.8490097848149845]|
|0    |1.0       |[0.1509902151850155,0.8490097848149845]|
|0    |1.0       |[0.1509902151850155,0.8490097848149845]|
|0    |1.0       |[0.1509902151850155,0.8490097848149845]|
|0    |1.0       |[0.1509902151850155,0.8490097848149845]|
|0    |1.0       |[0.1509902151850155,0.8490097848149845]|
|0    |1.0       |[0.1509902151850155,0.8490097848149845]|
|0    |1.0       |[0.1509902151850155,0.8490097848149845]|
|0    |1.0       |[0.1509902151850155,0.8490097848149845]|
|0    |1.0       |[0.1509902151850155,0.8490097848149845]|
+-----+----------+---------------------------------------+
only showing top 10 rows


## Decision Tree Structure

In [ ]:
print(best_dt.toDebugString)

DecisionTreeClassificationModel: uid=DecisionTreeClassifier_db81cd12a126, depth=0, numNodes=1, numClasses=2, numFeatures=2
  Predict: 1.0



# Model 3 – Random Forest Classifier

## Create Random Forest Model

In [ ]:
rf = RandomForestClassifier(
    featuresCol="scaledFeatures",
    labelCol="label",
    seed=42
)

## Hyperparameter Tuning

In [ ]:
paramGrid_rf = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [20, 50])
    .addGrid(rf.maxDepth, [5, 10])
    .build()
)

In [ ]:
tvs_rf = TrainValidationSplit(
    estimator=rf,
    estimatorParamMaps=paramGrid_rf,
    evaluator=binaryEvaluator,
    trainRatio=0.8
)

## Train Model

In [ ]:
# Random Forest Classifier Model

start = time.time()

rf_model = tvs_rf.fit(train_df)

end = time.time()

rf_training_time = end - start

print(f"Training Time: {rf_training_time:.2f} seconds")

Training Time: 2047.74 seconds


## Best Hyperparameters

In [ ]:
best_rf = rf_model.bestModel

print("Best Number of Trees :", best_rf.getNumTrees)
print("Best Max Depth :", best_rf.getMaxDepth())

Best Number of Trees : 20
Best Max Depth : 5


## Predictions

In [ ]:
rf_predictions = rf_model.transform(test_df)

## Accuracy

In [ ]:
accuracy_rf = accuracyEvaluator.evaluate(rf_predictions)

print("Accuracy =", accuracy_rf)

Accuracy = 0.8488031850233737


## Precision

In [ ]:
precision_rf = precisionEvaluator.evaluate(rf_predictions)

print("Precision =", precision_rf)

Precision = 0.7204668469058235


## Recall

In [ ]:
recall_rf = recallEvaluator.evaluate(rf_predictions)

print("Recall =", recall_rf)

Recall = 0.8488031850233737


## F1 Score

In [ ]:
f1_rf = f1Evaluator.evaluate(rf_predictions)

print("F1 Score =", f1_rf)

F1 Score = 0.7793872844249942


## AUC

In [ ]:
auc_rf = binaryEvaluator.evaluate(rf_predictions)

print("AUC =", auc_rf)

AUC = 0.5


## Sample Predictions

In [ ]:
rf_predictions.select(
    "label",
    "prediction",
    "probability"
).show(10, truncate=False)

+-----+----------+----------------------------------------+
|label|prediction|probability                             |
+-----+----------+----------------------------------------+
|1    |1.0       |[0.15100785502178254,0.8489921449782175]|
|0    |1.0       |[0.15100785502178254,0.8489921449782175]|
|0    |1.0       |[0.15100785502178254,0.8489921449782175]|
|0    |1.0       |[0.15100785502178254,0.8489921449782175]|
|0    |1.0       |[0.15100785502178254,0.8489921449782175]|
|0    |1.0       |[0.15100785502178254,0.8489921449782175]|
|0    |1.0       |[0.15100785502178254,0.8489921449782175]|
|0    |1.0       |[0.15100785502178254,0.8489921449782175]|
|0    |1.0       |[0.15100785502178254,0.8489921449782175]|
|0    |1.0       |[0.15100785502178254,0.8489921449782175]|
+-----+----------+----------------------------------------+
only showing top 10 rows


## Feature Importance

In [ ]:
print(best_rf.featureImportances)

(2,[],[])
